# VAW - pre-violence interaction understanding on the hand-picked clips
One notebook, run top to bottom (Runtime > Run all). It reads your start-time Excel, cuts every clip exactly at the start of the violence, and explains what happened between the people before it: distances, who approaches or follows whom, hands and arms, the last seconds, and how this differs from Normal clips. It also adds pretrained video models (InternVideo2 / X-CLIP / V-JEPA 2) next to the geometry.

**Before you start (once):** (1) Drive: *Shared with me > violence* > right-click > Organize > Add shortcut > My Drive. (2) Upload `FYP_data_filter.xlsx` to My Drive (or into `VAW_results`). (3) Runtime > Change runtime type > T4 GPU.

Everything is written to `VAW_results/curated_previolence/`. Every stage skips clips that are already done, so after a disconnect just run all again. A cell prints its summary only when it finishes: the progress lines (`prepared n/N`, `analyzed n/N`, `xclip: n/N clips`) and the Drive folder show it is working.

In [ ]:
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
url = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/KRITHIKSAI16/Violence-Against-Women.git"
if not os.path.isdir('/content/VAW'):
    !git clone -q {url} /content/VAW
else:
    !git -C /content/VAW pull -q {url} main
%cd /content/VAW
!pip install -q -r colab2/requirements.txt
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (switch the runtime to a T4 GPU: Runtime > Change runtime type)')
!python -m pytest -q tests/test_curated_index.py tests/test_curated_relation.py tests/test_curated_interaction.py tests/test_curated_evaluate.py

## 1. Find the videos and the start times

In [ ]:
# Where everything goes: a NEW sub-folder of VAW_results, so earlier results are not touched.
OUT = '/content/drive/MyDrive/VAW_results/curated_previolence'
# Optional manual paths (leave None to search Drive automatically):
DATA_ROOT = None        # e.g. '/content/drive/MyDrive/violence'
ANNOTATIONS = None      # e.g. '/content/drive/MyDrive/FYP_data_filter.xlsx'
from src.curated.locate import locate
data_root, ann, msgs = locate('/content/drive/MyDrive', '/content/drive/MyDrive/VAW_results', DATA_ROOT, ANNOTATIONS)
print('\n'.join(msgs))
args = "--out-root {} --data-root '{}' --annotations '{}'".format(OUT, data_root, ann)

## 2. Read the Excel, match the videos, cut every clip at its start time
Prints what matched and what did not (names in the sheet that have no video, videos with no row). Fix those first if the list is long; nothing is dropped silently.

In [ ]:
!python -m src.curated.run --stage prepare {args}

## 3. Pick the person detector by measurement (yolov8n vs yolo26x at 1280 px with BoT-SORT)
Runs both on a sample of your clips and keeps the one that finds more usable pairs. The comparison table is printed; the choice is saved in `perception_choice.json`. Skip with `--skip-perception` in the line above if you are short of time.

In [ ]:
!python -m src.curated.run --stage perception --out-root {OUT}

## 4. Shots and tracking of every clip (longest step on a GPU: progress counters appear in the log)

In [ ]:
!python -m src.curated.run --stage track --out-root {OUT}

## 5. Camera, distances in meters, scene layout and depth, interaction graph

In [ ]:
!python -m src.curated.run --stage context --out-root {OUT}

## 6. The understanding of each clip: sentences, distance figures, the pair videos (cut at the violence start)

In [ ]:
!python -m src.curated.run --stage analyze --out-root {OUT}

## 7. Video models (frozen InternVideo2 / X-CLIP / V-JEPA 2): zero-shot prompt scores and a leave-one-clip-out head
A model that cannot be loaded is named in the output and skipped; the others still run. The table shows every score next to the filming-style baseline.

In [ ]:
!python -m src.curated.run --stage ml --out-root {OUT}

## 8. Evaluation and the page for the jury

In [ ]:
!python -m src.curated.run --stage report --out-root {OUT}

## 9. Look at it here
The full page is `curated_previolence/report/index.html` on Drive (download the folder or the `curated_previolence_bundle.zip`, open `report/index.html`).

In [ ]:
import glob
from IPython.display import Image, display, HTML, Video
for f in sorted(glob.glob(OUT + '/figures/*/*_distance.png'))[:4]:
    print(f); display(Image(f))
print(open(OUT + '/evaluation.md').read())